<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/experiment/distilbert-lora-week2/hotel-review-nlp/notebooks/04_distilbert_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DistilBERT: from-scratch LoRA και full fine-tuning

Πρώτο GPU πείραμα της εβδομάδας 2: επαλήθευση της δικής μας LoRA υλοποίησης, εκπαίδευση adapters μαζί με την task head, full fine-tuning και ελεγχόμενη σύγκριση trainable parameters / macro-F1.

## 1. Στόχος και σειρά εκτέλεσης

Εκτέλεσε τα κελιά από πάνω προς τα κάτω σε **καθαρό Colab runtime με T4 GPU**. Ξεκίνα με `MODE = "smoke"`. Όταν περάσουν όλοι οι έλεγχοι, άλλαξε σε `MODE = "compare"` και χρησιμοποίησε νέο `EXPERIMENT`.

- Το notebook κάνει clone το ακριβές experiment branch· δεν χρειάζεται ZIP upload.
- Τα τρία frozen parquet splits επαναχρησιμοποιούνται όπως στο BiLSTM notebook.
- LoRA και full FT βλέπουν ακριβώς τα ίδια, με την ίδια σειρά, δεδομένα.
- Το dev macro-F1 επιλέγει checkpoint. Το test χρησιμοποιείται μόνο για την τελική σύγκριση.
- Τα smoke metrics ελέγχουν μόνο ότι η ροή λειτουργεί και δεν αποτελούν benchmark.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/krimits/hotel-review-nlp.git"
BRANCH = "experiment/distilbert-lora-week2"
CHECKOUT_DIR = Path("/content/hotel-review-distilbert-lora")

if not CHECKOUT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(CHECKOUT_DIR)],
        check=True,
    )

PROJECT_DIR = CHECKOUT_DIR / "hotel-review-nlp"
REPO_DIR = PROJECT_DIR
assert (PROJECT_DIR / "pyproject.toml").exists(), PROJECT_DIR
os.chdir(PROJECT_DIR)

# Keep Colab's CUDA-enabled torch and install the pinned experiment interfaces.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    check=True,
)
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

print("Project:", PROJECT_DIR)
print("Branch:", BRANCH)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

### Έλεγχος GPU και περιβάλλοντος

Το setup διατηρεί το CUDA PyTorch του Colab και εγκαθιστά τις εκδόσεις `transformers`/`peft` που ελέγχθηκαν για αυτό το πείραμα. Αν το Colab ζητήσει restart, κάνε restart και ξανατρέξε από την αρχή.

In [ ]:
import hashlib
import importlib.metadata as metadata
import json

import numpy as np
import pandas as pd
import peft
import torch
import transformers
import yaml
from IPython.display import display

assert torch.cuda.is_available(), "Select a Colab runtime with GPU."
assert transformers.__version__ == "4.56.2", transformers.__version__
assert peft.__version__ == "0.17.1", peft.__version__

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("Training dtype:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")

## 2. Ρυθμίσεις πειράματος και frozen δεδομένα

| Mode | Train | Dev | Test | Χρήση |
|---|---:|---:|---:|---|
| `smoke` | έως 200 | έως 100 | έως 100 | Γρήγορος έλεγχος ροής, 1 epoch |
| `compare` | έως 20.000 | έως 2.000 | ολόκληρο test | Πρώτη δίκαιη LoRA–full FT σύγκριση |
| `full` | ολόκληρο | ολόκληρο | ολόκληρο | Τελική εκτέλεση |

Το επόμενο κελί ψάχνει πρώτα τα ακριβή splits του προηγούμενου BiLSTM runtime και στο Drive. Αν δεν τα βρει, ζητά upload των `train.parquet`, `dev.parquet`, `test.parquet`. Δεν αναδημιουργεί splits με νέο seed.

In [ ]:
import shutil

USE_DRIVE = True
MODE = "smoke"  # smoke | compare | full
EXPERIMENT = "hotel_distilbert_smoke_v1"
SEED = 42

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_STORAGE = Path("/content/drive/MyDrive/hotel-review-nlp")
else:
    PROJECT_STORAGE = Path("/content/hotel-review-nlp-storage")

SOURCE_DATA = PROJECT_STORAGE / "data" / "processed"
SOURCE_DATA.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR = PROJECT_STORAGE / "experiments" / EXPERIMENT
EXPERIMENT_DATA = EXPERIMENT_DIR / "data"
CAPS = {
    "smoke": {"train": 200, "dev": 100, "test": 100},
    "compare": {"train": 20_000, "dev": 2_000, "test": None},
    "full": {"train": None, "dev": None, "test": None},
}
assert MODE in CAPS

required_files = ("train.parquet", "dev.parquet", "test.parquet")
candidate_directories = [
    PROJECT_STORAGE / "data" / "processed",
    Path("/content/hotel-review-threshold/hotel-review-nlp/data/processed"),
    Path("/content/hotel-review-nlp/hotel-review-nlp/data/processed"),
    Path("/content/data/processed"),
]

if not all((SOURCE_DATA / name).exists() for name in required_files):
    for candidate in candidate_directories:
        if candidate.resolve() == SOURCE_DATA.resolve():
            continue
        if all((candidate / name).exists() for name in required_files):
            for name in required_files:
                shutil.copy2(candidate / name, SOURCE_DATA / name)
            print("Copied frozen splits from:", candidate)
            break

missing = [name for name in required_files if not (SOURCE_DATA / name).exists()]
if missing:
    from google.colab import files

    print("Upload the exact frozen split files:", missing)
    uploaded = files.upload()
    for uploaded_name, content in uploaded.items():
        basename = Path(uploaded_name).name
        if basename in required_files:
            (SOURCE_DATA / basename).write_bytes(content)

missing = [name for name in required_files if not (SOURCE_DATA / name).exists()]
if missing:
    raise FileNotFoundError(f"Still missing frozen split files: {missing}")

expected_source = {
    "train": {"rows": 118_990, "negative": 26_232, "positive": 92_758},
    "dev": {"rows": 14_872, "negative": 3_278, "positive": 11_594},
    "test": {"rows": 13_278, "negative": 3_278, "positive": 10_000},
}
source_summary = []
for split, expected in expected_source.items():
    frame = pd.read_parquet(SOURCE_DATA / f"{split}.parquet")
    counts = frame["label"].value_counts().to_dict()
    observed = {
        "rows": len(frame),
        "negative": int(counts.get("negative", 0)),
        "positive": int(counts.get("positive", 0)),
    }
    assert observed == expected, (split, observed, expected)
    source_summary.append({"split": split, **observed})

display(pd.DataFrame(source_summary))
print("Frozen source splits:", SOURCE_DATA)
print("Experiment artifacts:", EXPERIMENT_DIR)

### Δημιουργία και ταυτοποίηση των experiment splits

Τα caps εφαρμόζονται ντετερμινιστικά και stratified. Αποθηκεύεται order-sensitive SHA-256 για κάθε split, ώστε το notebook να αρνείται σύγκριση αν LoRA και full FT δεν αξιολογήθηκαν στα ίδια ακριβώς δεδομένα.

In [ ]:
from reviewnlp.utils.experiments import prepare_experiment_splits

manifest = prepare_experiment_splits(SOURCE_DATA, EXPERIMENT_DATA, CAPS[MODE], SEED)
frames = {
    split: pd.read_parquet(EXPERIMENT_DATA / f"{split}.parquet")
    for split in ("train", "dev", "test")
}
display(
    pd.DataFrame(
        {
            split: {"rows": len(frame), **frame["label"].value_counts().to_dict()}
            for split, frame in frames.items()
        }
    ).T
)
display(frames["train"].head(3))

source_hashes = {
    str(path.relative_to(REPO_DIR)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted((REPO_DIR / "src").rglob("*.py"))
}
environment = {
    "python": sys.version,
    "gpu": torch.cuda.get_device_name(0),
    "mode": MODE,
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip(),
    "packages": {
        name: metadata.version(name)
        for name in [
            "torch",
            "transformers",
            "peft",
            "accelerate",
            "pandas",
            "numpy",
            "scikit-learn",
        ]
    },
    "source_sha256": source_hashes,
}
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
environment_path = EXPERIMENT_DIR / "environment.json"
if environment_path.exists():
    previous = json.loads(environment_path.read_text())
    assert previous == environment, "Environment changed: choose a new EXPERIMENT."
else:
    environment_path.write_text(json.dumps(environment, indent=2), encoding="utf-8")

### Βοηθητική εκτέλεση CLI

Τα training commands τρέχουν σε ξεχωριστή διεργασία. Έτσι ελευθερώνεται η GPU RAM
όταν τελειώνει κάθε μοντέλο και αποθηκεύεται το πλήρες log στο Drive.
Επιτυχημένο run επαναχρησιμοποιείται μόνο με τις ίδιες ρυθμίσεις.
Αν το Colab διακοπεί πριν από το τελικό checkpoint, ξανατρέξε το κελί εκπαίδευσης
από την αρχή· δεν υπάρχει resume optimizer από ενδιάμεσα checkpoints.

In [ ]:
def run_cli(module, config, output_dir, extra_args=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    config_path = output_dir / "run_config.yaml"
    request = {"module": module, "config": config, "extra_args": extra_args or []}
    request_path = output_dir / "request.json"
    if request_path.exists():
        assert json.loads(request_path.read_text()) == request, "Settings changed: use a new EXPERIMENT."
    else:
        request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    if (output_dir / "metrics.json").exists():
        print("Using completed run:", output_dir)
        return
    command = [sys.executable, "-u", "-m", module, "--config", str(config_path), *(extra_args or [])]
    print(" ".join(command))
    with (output_dir / "train.log").open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Training failed ({status}); see {output_dir / 'train.log'}")

## 3. Επαλήθευση LoRA πριν από το GPU training

Η ενημέρωση είναι $\Delta W=(\alpha/r)BA$. Τα pretrained βάρη παγώνουν, ενώ εκπαιδεύονται οι πίνακες A/B και η νέα task head. Το PEFT equivalence test δημιουργεί μικρό BERT τοπικά, αντιγράφει τα ίδια adapters στις δύο υλοποιήσεις και απαιτεί αριθμητικά ίδιες εξόδους.

In [ ]:
# Equivalent local command: pytest tests/test_lora.py -v
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_lora.py", "-v", "-o", "addopts="],
    check=True,
)

## 4. Ρυθμίσεις DistilBERT

Full FT και LoRA μοιράζονται tokenizer, sequence length, seed, batches, epochs και splits. Το learning rate καταγράφεται: `2e-5` για full FT και `1e-4` για LoRA. Το checkpoint επιλέγεται αποκλειστικά με dev macro-F1.

Στο LoRA run εκπαιδεύονται οι adapters στα `q_lin`/`v_lin` **και** οι `pre_classifier`/`classifier`, επειδή η classification head είναι νέα και τυχαία αρχικοποιημένη. Το πάγωμά της θα καθιστούσε τη σύγκριση άκυρη.

In [ ]:
FULL_DIR = EXPERIMENT_DIR / "distilbert"
LORA_DIR = EXPERIMENT_DIR / "distilbert_lora_scratch"
DISTILBERT_EPOCHS = 1 if MODE == "smoke" else 2
DISTILBERT_BATCH = 16
LORA_RANK = 8
LORA_ALPHA = 16
LORA_LR = 1e-4
encoder_config = {
    "seed": SEED,
    "model": {"name": "distilbert-base-uncased", "max_length": 256},
    "data": {"processed_dir": str(EXPERIMENT_DATA)},
    "train": {
        "batch_size": DISTILBERT_BATCH, "eval_batch_size": 32,
        "epochs": DISTILBERT_EPOCHS, "lr": 2e-5, "weight_decay": 0.01,
        "warmup_ratio": 0.06, "fp16": True,
    },
    "output": {"model_dir": str(FULL_DIR)},
}
print(yaml.safe_dump(encoder_config, sort_keys=False))

## 5. Πρώτο GPU run: from-scratch LoRA

Καλούμε το module reviewnlp.llm.train_distilbert_lora.
Στο log έλεγξε τα injections στα q_lin, v_lin, τις trainable παραμέτρους και
ότι το loss είναι πεπερασμένο.
Η προειδοποίηση για νέα classification weights κατά την πρώτη φόρτωση είναι αναμενόμενη.

In [ ]:
run_cli(
    "reviewnlp.llm.train_distilbert_lora", encoder_config, LORA_DIR,
    ["--r", str(LORA_RANK), "--alpha", str(LORA_ALPHA), "--lr", str(LORA_LR)],
)

## 6. Full fine-tuning με τα ίδια splits

Το δεύτερο run ξεκινά ξανά από το ίδιο pretrained DistilBERT.
Όλες οι παράμετροι εκπαιδεύονται.

In [ ]:
run_cli("reviewnlp.llm.train_distilbert", encoder_config, FULL_DIR)

## 7. Σύγκριση trainable parameters και test macro-F1

Το κελί αρνείται σύγκριση αν άλλαξε έστω και η σειρά των δεδομένων.
Το test χρησιμοποιείται για την τελική αναφορά, όχι για επιλογή rank/LR/epochs.
Για αλλαγές υπερπαραμέτρων χρησιμοποίησε το dev και κατόπιν κλείδωσε τις επιλογές.
Ένα seed δίνει αρχική ένδειξη, όχι στατιστικά βέβαιο αποτέλεσμα.

In [ ]:
full = json.loads((FULL_DIR / "metrics.json").read_text())
lora = json.loads((LORA_DIR / "metrics.json").read_text())
assert full["data"] == lora["data"] == manifest["splits"]
np.testing.assert_array_equal(
    np.load(FULL_DIR / "test_labels.npy"), np.load(LORA_DIR / "test_labels.npy")
)
comparison = pd.DataFrame([
    {
        "model": name, "mode": MODE,
        "trainable_params": metrics["trainable_params"],
        "adapter_params": metrics["adapter_params"],
        "head_params": metrics["head_params"],
        "total_params": metrics["total_params"],
        "trainable_pct": 100 * metrics["trainable_params"] / metrics["total_params"],
        "dev_macro_f1": metrics["best_dev_macro_f1"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "training_minutes": metrics["training_seconds"] / 60,
        "peak_cuda_mb": metrics["peak_cuda_memory_mb"],
        "learning_rate": metrics["settings"]["lr"],
    }
    for name, metrics in [("DistilBERT full FT", full), ("DistilBERT scratch LoRA + head", lora)]
])
display(comparison.round(4))
reduction = 100 * (1 - lora["trainable_params"] / full["trainable_params"])
print("Trainable reduction vs full FT:", f"{reduction:.2f}%")
print("LoRA minus full FT macro-F1:", round(lora["test"]["macro_f1"] - full["test"]["macro_f1"], 4))
comparison.to_csv(EXPERIMENT_DIR / "distilbert_comparison.csv", index=False)

### Έλεγχος επαναφόρτωσης του LoRA μοντέλου

Το μικρό `lora_scratch.pt` κρατά adapters και task head. Επιπλέον αποθηκεύεται merged checkpoint σε κανονικό Hugging Face format, ώστε το inference να μη χρειάζεται custom wrapper. Το επόμενο κελί επαληθεύει ότι η επαναφόρτωση αναπαράγει τις αποθηκευμένες προβλέψεις.

In [ ]:
from reviewnlp.llm.predict import predict_encoder

sample = frames["test"].head(16)
reloaded = predict_encoder(str(LORA_DIR), sample["text"].tolist(), max_length=256, batch_size=8)
cached_ids = np.load(LORA_DIR / "test_logits.npy")[:len(sample)].argmax(axis=-1)
cached_labels = np.array(["negative", "positive"])[cached_ids]
np.testing.assert_array_equal(reloaded, cached_labels)
print("Merged checkpoint reload: PASS")
display(pd.DataFrame({
    "review": sample["text"].tolist(), "gold": sample["label"].tolist(), "prediction": reloaded,
}).head(5))

## 8. Επόμενο βήμα

1. Εκτέλεσε πρώτα το `smoke` run.
2. Άλλαξε σε `MODE = "compare"` και νέο `EXPERIMENT`, π.χ. `hotel_distilbert_compare_v1`.
3. Εκτέλεσε ξανά από την αρχή και κράτησε το `distilbert_comparison.csv`, τα δύο `metrics.json`, configs και logs από το Drive.
4. Στείλε τα αποτελέσματα για να ενημερώσουμε το README και να κλείσουμε την εβδομάδα 2.

Το `02_train_qlora_colab.ipynb` αποτελεί το επόμενο, ξεχωριστό QLoRA πείραμα.